# Heart Disease Risk Prediction — Data Mining Analysis

**Framework:** CRISP-DM  
**Dataset:** Heart Failure Prediction (Kaggle) — 918 pasien, 11 fitur  
**Metode:** Classification (Logistic Regression, Random Forest, XGBoost) + Clustering (K-Means) + SHAP Explainability

---

## 0. Setup & Imports

In [ ]:
import sys
import os

# Add project root and app/ to path
NOTEBOOK_DIR = os.path.abspath('.')
ROOT = os.path.join(NOTEBOOK_DIR, '..')
APP_DIR = os.path.join(ROOT, 'app')
for p in (ROOT, APP_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
matplotlib.rcParams['figure.figsize'] = (10, 5)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False
plt.style.use('seaborn-v0_8-whitegrid')

DATASET_PATH = os.path.join(ROOT, 'dataset', 'heart.csv')
MODEL_DIR    = os.path.join(ROOT, 'model')
os.makedirs(MODEL_DIR, exist_ok=True)

print('Python:', sys.version)
print('Pandas:', pd.__version__)
print('NumPy:', np.__version__)
print('Dataset:', DATASET_PATH)
print('Model output:', MODEL_DIR)

---
## 1. Business Understanding

Penyakit jantung adalah penyebab kematian nomor satu secara global. Tujuan proyek ini:

- **Klasifikasi**: Membangun model prediktif untuk mendeteksi risiko penyakit jantung secara dini.
- **Segmentasi**: Mengelompokkan pasien berdasarkan profil klinis menggunakan K-Means.
- **Interpretasi**: Menjelaskan kontribusi setiap fitur terhadap prediksi menggunakan SHAP.

**Target variabel**: `HeartDisease` (1 = penyakit jantung terdeteksi, 0 = tidak)

---
## 2. Data Understanding

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('=== Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Cholesterol == 0 (anomali) ===')
print(f"{(df['Cholesterol'] == 0).sum()} baris")

In [ ]:
df.describe().round(2)

In [ ]:
# Distribusi kelas target
counts = df['HeartDisease'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['No Disease (0)', 'Heart Disease (1)'], counts.values,
            color=['#2B2D42', '#E84545'], edgecolor='white', linewidth=0.8)
axes[0].set_title('Distribusi Kelas Target')
axes[0].set_ylabel('Jumlah Pasien')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['No Disease', 'Heart Disease'],
            colors=['#2B2D42', '#E84545'], autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Proporsi Kelas')

plt.suptitle('Class Balance', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Distribusi fitur numerik
numeric_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    for label, color in [(0, '#2B2D42'), (1, '#E84545')]:
        subset = df[df['HeartDisease'] == label][col]
        axes[i].hist(subset, bins=30, alpha=0.6, color=color,
                     label=f'HeartDisease={label}', edgecolor='none')
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

axes[-1].axis('off')
plt.suptitle('Distribusi Fitur Numerik berdasarkan Target', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distribusi fitur kategorikal
cat_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ct = df.groupby([col, 'HeartDisease']).size().unstack(fill_value=0)
    ct.plot(kind='bar', ax=axes[i], color=['#2B2D42', '#E84545'],
            edgecolor='white', linewidth=0.5, rot=0)
    axes[i].set_title(col)
    axes[i].set_xlabel('')
    axes[i].legend(['No Disease', 'Heart Disease'], fontsize=8)

axes[-1].axis('off')
plt.suptitle('Distribusi Fitur Kategorikal berdasarkan Target', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
num_df = df[['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak', 'HeartDisease']]
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, ax=ax, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Matriks Korelasi Fitur Numerik', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots
fig, axes = plt.subplots(1, 5, figsize=(16, 5))
for i, col in enumerate(numeric_cols):
    data = [df[df['HeartDisease']==0][col].dropna(),
            df[df['HeartDisease']==1][col].dropna()]
    bp = axes[i].boxplot(data, patch_artist=True,
                         medianprops={'color': 'white', 'linewidth': 2})
    for patch, color in zip(bp['boxes'], ['#2B2D42', '#E84545']):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    axes[i].set_title(col)
    axes[i].set_xticks([1, 2])
    axes[i].set_xticklabels(['No\nDisease', 'Heart\nDisease'], fontsize=8)

plt.suptitle('Box Plot Fitur Numerik vs HeartDisease', fontweight='bold')
plt.tight_layout()
plt.show()

**Temuan EDA:**
- `Cholesterol` memiliki nilai 0 yang tidak valid (akan diimputasi dengan median).
- `Oldpeak` dan `MaxHR` menunjukkan perbedaan distribusi yang signifikan antara kelas.
- `ChestPainType=ASY` dan `ExerciseAngina=Y` sangat berkorelasi dengan penyakit jantung.
- Dataset relatif seimbang (55% vs 45%).

---
## 3. Data Preparation

In [ ]:
from src.data.loader import load_raw_data
from src.data.preprocessor import full_pipeline
from src.utils.io_helpers import save_model, save_json, save_numpy

df_raw = load_raw_data(DATASET_PATH)
X_train, X_test, y_train, y_test, scaler, feature_columns, chol_median = full_pipeline(df_raw)

print(f'Train set : {X_train.shape}')
print(f'Test set  : {X_test.shape}')
print(f'Features  : {feature_columns}')
print(f'Cholesterol zero median : {chol_median:.2f}')

# Save scaler & metadata
save_model(scaler, os.path.join(MODEL_DIR, 'scaler.joblib'))
save_json({'feature_columns': feature_columns, 'chol_median': chol_median},
          os.path.join(MODEL_DIR, 'feature_columns.json'))
save_numpy(X_train, os.path.join(MODEL_DIR, 'X_train.npy'))
save_numpy(X_test,  os.path.join(MODEL_DIR, 'X_test.npy'))
save_numpy(y_test,  os.path.join(MODEL_DIR, 'y_test.npy'))
print('\nArtifacts saved: scaler.joblib, feature_columns.json, X_train/test.npy')

In [ ]:
# Visualisasi distribusi sebelum vs sesudah imputasi Cholesterol
df_clean = df_raw.copy()
df_clean.loc[df_clean['Cholesterol'] == 0, 'Cholesterol'] = chol_median

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_raw['Cholesterol'], bins=40, color='#2B2D42', alpha=0.8, edgecolor='none')
axes[0].set_title('Cholesterol — Sebelum Imputasi')
axes[0].set_xlabel('Cholesterol (mm/dl)')
axes[1].hist(df_clean['Cholesterol'], bins=40, color='#E84545', alpha=0.8, edgecolor='none')
axes[1].set_title('Cholesterol — Setelah Imputasi Median')
axes[1].set_xlabel('Cholesterol (mm/dl)')
plt.tight_layout()
plt.show()

---
## 4. Modeling — Klasifikasi

In [ ]:
from src.models.trainer import train_all

print('Training models (GridSearchCV)...')
models = train_all(X_train, y_train)

for name, model in models.items():
    save_model(model, os.path.join(MODEL_DIR, f'{name}.joblib'))
    print(f'  Saved: {name}.joblib')

---
## 5. Evaluation — Klasifikasi

In [ ]:
from src.models.evaluator import evaluate_all, get_feature_importances

results = evaluate_all(models, X_test, y_test)
save_json(results, os.path.join(MODEL_DIR, 'eval_results.json'))

# Summary table
rows = []
for key, name in [('logistic_regression', 'Logistic Regression'),
                   ('random_forest', 'Random Forest'),
                   ('xgboost', 'XGBoost')]:
    r = results[key]
    rows.append({'Model': name, 'Accuracy': r['accuracy'],
                 'F1-Score': r['f1'], 'ROC-AUC': r['roc_auc'],
                 'Precision': r['precision'], 'Recall': r['recall']})

eval_df = pd.DataFrame(rows).set_index('Model')
eval_df.round(4)

In [ ]:
# Bar chart perbandingan metrik
metrics = ['accuracy', 'f1', 'roc_auc']
metric_labels = ['Accuracy', 'F1-Score', 'ROC-AUC']
model_keys = ['logistic_regression', 'random_forest', 'xgboost']
model_names = ['Logistic\nRegression', 'Random\nForest', 'XGBoost']
colors = ['#636EFA', '#00CC96', '#EF553B']

x = np.arange(len(model_names))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5))

for i, (m, label) in enumerate(zip(metrics, metric_labels)):
    vals = [results[k][m] for k in model_keys]
    bars = ax.bar(x + i*width, vals, width, label=label, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(model_names)
ax.set_ylim(0.7, 1.0)
ax.set_ylabel('Score')
ax.set_title('Perbandingan Metrik Klasifikasi', fontweight='bold', pad=12)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix & ROC curve untuk setiap model
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
labels_cm = ['No Disease', 'Heart Disease']

for col, (key, name) in enumerate(zip(model_keys, ['Logistic Regression', 'Random Forest', 'XGBoost'])):
    r = results[key]
    cm = np.array(r['confusion_matrix'])

    # Confusion Matrix
    im = axes[0, col].imshow(cm, cmap='Blues', aspect='auto')
    axes[0, col].set_xticks([0, 1]); axes[0, col].set_yticks([0, 1])
    axes[0, col].set_xticklabels(labels_cm, fontsize=8)
    axes[0, col].set_yticklabels(labels_cm, fontsize=8)
    axes[0, col].set_xlabel('Predicted'); axes[0, col].set_ylabel('Actual')
    axes[0, col].set_title(f'{name}\nConfusion Matrix', fontweight='bold', fontsize=9)
    for i in range(2):
        for j in range(2):
            axes[0, col].text(j, i, str(cm[i, j]), ha='center', va='center',
                              color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=14, fontweight='bold')

    # ROC Curve
    axes[1, col].plot(r['fpr'], r['tpr'], color='#E84545', lw=2,
                      label=f"AUC = {r['roc_auc']:.4f}")
    axes[1, col].plot([0,1], [0,1], 'k--', lw=1, alpha=0.5)
    axes[1, col].set_xlabel('False Positive Rate')
    axes[1, col].set_ylabel('True Positive Rate')
    axes[1, col].set_title(f'{name}\nROC Curve', fontweight='bold', fontsize=9)
    axes[1, col].legend(fontsize=9)

plt.suptitle('Evaluasi Model — Confusion Matrix & ROC Curve', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for ax, (key, name) in zip(axes, zip(model_keys, ['Logistic Regression', 'Random Forest', 'XGBoost'])):
    fi_df = get_feature_importances(models[key], feature_columns).head(12)
    ax.barh(fi_df['feature'], fi_df['importance'], color='#E84545', alpha=0.8)
    ax.invert_yaxis()
    ax.set_title(f'{name}', fontweight='bold')
    ax.set_xlabel('Importance')
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('Feature Importance — Top 12 Fitur per Model', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Modeling — Clustering K-Means

In [ ]:
from src.models.clustering import (
    compute_elbow_data, compute_silhouette_scores,
    get_optimal_k, run_kmeans, get_pca_projection, get_cluster_profiles,
)

X_all_scaled = np.vstack([X_train, X_test])

elbow    = compute_elbow_data(X_all_scaled)
sil      = compute_silhouette_scores(X_all_scaled)
optimal_k = get_optimal_k(sil)

print(f'Optimal k (silhouette): {optimal_k}')

# Elbow & Silhouette plots
sil_df = pd.DataFrame(sil)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(elbow['k'], elbow['inertia'], 'o-', color='#E84545', lw=2)
axes[0].set_xlabel('k (jumlah cluster)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Curve', fontweight='bold')

axes[1].plot(sil_df['k'], sil_df['score'], 'o-', color='#2B2D42', lw=2)
opt_score = sil_df.loc[sil_df['k'] == optimal_k, 'score'].values[0]
axes[1].scatter([optimal_k], [opt_score], color='red', s=120, zorder=5,
                label=f'Optimal k={optimal_k}')
axes[1].set_xlabel('k (jumlah cluster)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs k', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Fit K-Means dengan optimal k
km_model, km_labels = run_kmeans(X_all_scaled, optimal_k)

save_model(km_model, os.path.join(MODEL_DIR, 'kmeans.joblib'))
save_json({'elbow': elbow, 'silhouette': sil, 'optimal_k': optimal_k},
          os.path.join(MODEL_DIR, 'clustering_data.json'))
save_numpy(km_labels, os.path.join(MODEL_DIR, 'kmeans_labels.npy'))
save_numpy(X_all_scaled, os.path.join(MODEL_DIR, 'X_all_scaled.npy'))

print(f'K-Means fitted with k={optimal_k}')
print(f'Label distribution: {dict(zip(*np.unique(km_labels, return_counts=True)))}')

In [ ]:
# PCA 2D visualization
pca_coords = get_pca_projection(X_all_scaled)

fig, ax = plt.subplots(figsize=(9, 6))
palette = ['#2B2D42', '#E84545', '#00CC96', '#FF7F0E']
for k in range(optimal_k):
    mask = km_labels == k
    ax.scatter(pca_coords[mask, 0], pca_coords[mask, 1],
               s=18, alpha=0.65, label=f'Cluster {k}',
               color=palette[k % len(palette)])

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title(f'Distribusi Cluster K-Means (k={optimal_k}) — PCA 2D', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cluster profiles
numeric_cols_raw = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak', 'HeartDisease']
df_num = df_raw.copy()
df_num.loc[df_num['Cholesterol'] == 0, 'Cholesterol'] = chol_median
df_num_subset = df_num[numeric_cols_raw].copy()

n_rows   = len(df_num_subset)
n_labels = len(km_labels)
labels_subset = km_labels[:n_rows] if n_labels >= n_rows else km_labels

profiles = get_cluster_profiles(df_num_subset, labels_subset)
print('Profil Cluster:')
profiles

In [ ]:
# HeartDisease prevalence per cluster
df_num_subset_copy = df_num_subset.copy()
df_num_subset_copy['Cluster'] = labels_subset
prevalence = df_num_subset_copy.groupby('Cluster')['HeartDisease'].mean()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([f'Cluster {i}' for i in prevalence.index],
              prevalence.values, color=palette[:len(prevalence)],
              edgecolor='white', alpha=0.85)
for bar, v in zip(bars, prevalence.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{v:.1%}', ha='center', fontweight='bold')
ax.set_ylabel('HeartDisease Prevalence')
ax.set_title('Prevalensi Penyakit Jantung per Cluster', fontweight='bold')
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

---
## 7. SHAP Explainability

In [ ]:
from src.explainability.shap_explainer import get_explainer, compute_shap_values, get_summary_data
import shap

xgb_model = models['xgboost']
explainer  = get_explainer(xgb_model, X_train[:100])
shap_vals  = compute_shap_values(explainer, X_test)

save_numpy(shap_vals, os.path.join(MODEL_DIR, 'shap_values_xgboost.npy'))
print(f'SHAP values shape: {shap_vals.shape}')

In [ ]:
# Global: Mean |SHAP| bar chart
summary = get_summary_data(shap_vals, feature_columns)

fig, ax = plt.subplots(figsize=(8, 6))
top_n = 15
feats = summary['features'][:top_n]
vals  = summary['mean_abs_shap'][:top_n]

bars = ax.barh(feats, vals, color='#E84545', alpha=0.8)
ax.invert_yaxis()
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title('Global Feature Importance (XGBoost) — Mean |SHAP|', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Beeswarm plot
base_val = explainer.expected_value
if isinstance(base_val, (list, np.ndarray)):
    base_val = float(base_val[1])

explanation = shap.Explanation(
    values=shap_vals,
    base_values=np.full(len(shap_vals), base_val),
    data=X_test,
    feature_names=feature_columns,
)
plt.figure(figsize=(10, 6))
shap.plots.beeswarm(explanation, max_display=15, show=True)

In [ ]:
# Local: Waterfall untuk satu sampel
SAMPLE_IDX = 0

waterfall_exp = shap.Explanation(
    values=shap_vals[SAMPLE_IDX],
    base_values=base_val,
    data=X_test[SAMPLE_IDX],
    feature_names=feature_columns,
)
plt.figure(figsize=(10, 5))
shap.plots.waterfall(waterfall_exp, max_display=12, show=True)
print(f'Actual label: {"Heart Disease" if y_test[SAMPLE_IDX] == 1 else "No Disease"}')

---
## 8. Summary — Model Artifacts

In [ ]:
# Ringkasan hasil evaluasi
print('=' * 55)
print('HASIL EVALUASI MODEL')
print('=' * 55)
for key, name in zip(model_keys, ['Logistic Regression', 'Random Forest', 'XGBoost']):
    r = results[key]
    print(f'{name:22s}  F1={r["f1"]:.4f}  AUC={r["roc_auc"]:.4f}  Acc={r["accuracy"]:.4f}')

print()
print('ARTIFACTS TERSIMPAN DI model/')
print('=' * 55)
for f in sorted(os.listdir(MODEL_DIR)):
    size = os.path.getsize(os.path.join(MODEL_DIR, f))
    print(f'  {f:40s} {size/1024:.1f} KB')